### Mohammed Omar Mkhallati
### AI for Cybersecurity
### Monday, September 14, 2026
### CyberAttack Classifier
### Dataset Used: https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset

#### Import Statements

In [1]:
import ssl
import pandas as pd
import numpy as np

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

#### Reading Dataset From My GitHub

In [2]:
ssl._create_default_https_context = ssl._create_unverified_context
github_url = "https://raw.githubusercontent.com/omarmkhallati/AI_For_Cybersecurity-Mkhallati/a373179b1f5b8929c1716ddb82290f489beed8f8/CEAS_08.csv"
df = pd.read_csv(github_url)

#### Confirm the Dataset was Read Properly

In [3]:
df.head()

,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


#### Features

In [4]:
# Feature 1 : Text Data
df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
df['text'].head()

0    Never agree to be a loser Buck up, your troubl...
1    Befriend Jenna Jameson \nUpgrade your sex and ...
2    CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...
3    Re: svn commit: r619753 - in /spamassassin/tru...
4    SpecialPricesPharmMoreinfo \nWelcomeFastShippi...
Name: text, dtype: object

In [5]:
# Feature 2 : Sender Domain
def get_domain (sender) :
    if pd.isnull(sender):
        return ''
    else:
        domain = sender.split('@')[-1]
        domain = domain.split('>')[0]
        return domain

df['sender_domain'] = df['sender'].apply(get_domain)
df['sender_domain'].head()

0              iworld.de
1              icable.ph
2    universalnet.psi.br
3              pobox.com
4    loanofficertool.com
Name: sender_domain, dtype: object

In [6]:
# Feature 3 : URL Presence
df['urls'] = df['urls'].astype(int)
df['urls'].head()

0    1
1    1
2    1
3    1
4    1
Name: urls, dtype: int64

#### Splitting the Data Into Training and Testing

In [14]:
X = df[['text', 'sender_domain', 'urls']]
Y = df['label']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.4, random_state=42)

#### Turning the Email Text Into Numbers (AI Helped Me With This Concept)

In [15]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_text = vectorizer.fit_transform(X_train["text"])
X_test_text = vectorizer.transform(X_test["text"])

#### Turning the Sender Domain Into a Number

In [16]:
domain_rates =Y_train.groupby(X_train["sender_domain"]).mean()
global_rate = Y_train.mean()
X_train_domain = X_train["sender_domain"].map(domain_rates).fillna(global_rate).values.reshape(-1, 1)
X_test_domain = X_test["sender_domain"].map(domain_rates).fillna(global_rate).values.reshape(-1, 1)

#### The URL Presence

In [17]:
X_train_urls = X_train["urls"].values.reshape(-1, 1)
X_test_urls = X_test["urls"].values.reshape(-1, 1)

#### Putting All 3 Features Together After Making Them All Numbers

In [18]:
X_train_final = hstack([X_train_text, csr_matrix(X_train_domain), csr_matrix(X_train_urls)])
X_test_final = hstack([X_test_text, csr_matrix(X_test_domain), csr_matrix(X_test_urls)])

#### Training and Testing The Model

In [19]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_final, Y_train)
Y_pred = model.predict(X_test_final)

#### Accuracy Results

In [20]:
print("Accuracy: ", accuracy_score(Y_test, Y_pred))
print("Precision:", precision_score(Y_test, Y_pred))
print("Recall:   ", recall_score(Y_test, Y_pred))
print("F1 score: ", f1_score(Y_test, Y_pred))
print(confusion_matrix(Y_test, Y_pred))

Accuracy:  0.9788660452049547
Precision: 0.9712230215827338
Recall:    0.9913941480206541
F1 score:  0.9812049287377207
[[6691  256]
 [  75 8640]]


### Of the 15,662 test emails, the model correctly identified 6,691 legitimate emails and 8,640 phishing emails. It made 256 false positives (legitimate emails wrongly flagged as phishing) and only 75 false negatives (phishing emails that slipped through as legitimate).